In [0]:
%sql
-- Criando uma pasta segura (Volume) 
CREATE VOLUME IF NOT EXISTS projeto_bcb.bronze.arquivos_brutos;


In [0]:
import requests

# A URL que o banco central gerou
url = "https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/CotacaoMoedaPeriodo(moeda=@moeda,dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)?@moeda='USD'&@dataInicial='06-01-2026'&@dataFinalCotacao='06-30-2026'&$top=100&$filter=tipoBoletim%20eq%20'Fechamento'&$format=json&$select=cotacaoCompra,dataHoraCotacao"

response = requests.get(url)

if response.status_code == 200:
    dados_json = response.json()['value']
    df_bronze = spark.createDataFrame(dados_json)
    
    display(df_bronze)
    
    caminho_bronze = "/Volumes/projeto_bcb/bronze/arquivos_brutos/cotacoes_novos"
    # Transformando em parquet
    (df_bronze.write
       .mode("overwrite")
       .format("parquet")
       .save(caminho_bronze))
       
    print(f"\nSucesso! Arquivos Parquet salvos na camada Bronze: {caminho_bronze}")
else:
    print(f"Erro {response.status_code}")